In [7]:
import re
import joblib
import pandas as pd
import numpy as np
import nltk
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# ── Optional dependencies ────────────────────────────────────────────────────
try:
    import ftfy
except ImportError:
    ftfy = None

try:
    import emoji
except ImportError:
    emoji = None

try:
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
    base_stop_words = set(ENGLISH_STOP_WORDS)
except Exception:
    base_stop_words = set()

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

# ── Constants ────────────────────────────────────────────────────────────────
RANDOM_STATE = 42

negation_words = {
    "no", "not", "nor", "never", "none", "nothing", "nowhere", "neither",
    "cannot", "can't", "dont", "don't", "didnt", "didn't", "doesnt", "doesn't",
    "isnt", "isn't", "wasnt", "wasn't", "werent", "weren't", "wont", "won't",
    "wouldnt", "wouldn't", "shouldnt", "shouldn't", "couldnt", "couldn't"
}
safe_stop_words = base_stop_words - negation_words

slang_dict = {
    "u": "you", "ur": "your", "ya": "you", "idk": "i do not know",
    "imo": "in my opinion", "imho": "in my humble opinion",
    "btw": "by the way", "brb": "be right back",
    "lol": "laughing", "lmao": "laughing", "omg": "oh my god",
    "tf": "the fuck", "wtf": "what the fuck", "rn": "right now",
    "tho": "though", "thx": "thanks", "pls": "please", "plz": "please",
    "bc": "because", "bcz": "because", "cuz": "because",
    "gonna": "going to", "wanna": "want to", "gotta": "got to",
    "kinda": "kind of", "sorta": "sort of"
}

contraction_dict = {
    "i'm": "i am", "i've": "i have", "i'll": "i will", "i'd": "i would",
    "you're": "you are", "you've": "you have", "he's": "he is",
    "she's": "she is", "it's": "it is", "we're": "we are",
    "they're": "they are", "can't": "cannot", "won't": "will not",
    "don't": "do not", "didn't": "did not", "doesn't": "does not",
    "isn't": "is not", "aren't": "are not", "wasn't": "was not",
    "weren't": "were not", "shouldn't": "should not",
    "wouldn't": "would not", "couldn't": "could not"
}

# ── Compiled patterns ────────────────────────────────────────────────────────
url_pattern        = re.compile(r"https?://\S+|www\.\S+")
mention_pattern    = re.compile(r"@\w+")
hashtag_pattern    = re.compile(r"#(\w+)")
html_pattern       = re.compile(r"<.*?>")

# ── Helper functions ─────────────────────────────────────────────────────────
def fix_unicode_text(text):
    return ftfy.fix_text(text) if ftfy is not None else text

def convert_emojis(text):
    if emoji is not None:
        text = emoji.demojize(text, delimiters=(" ", " "))
        text = text.replace(":", " ").replace("_", " ")
    return text

def expand_contractions(text):
    return " ".join(contraction_dict.get(w, w) for w in text.split())

def normalize_slang_text(text):
    return " ".join(slang_dict.get(w, w) for w in text.split())

def reduce_elongated_words(text):
    return re.sub(r"(\w)\1{2,}", r"\1\1", text)

def reduce_repeated_punctuation(text):
    text = re.sub(r"!{2,}", "!", text)
    text = re.sub(r"\?{2,}", "?", text)
    text = re.sub(r"\.{3,}", "...", text)
    return text

def remove_special_characters(text):
    return re.sub(r"[^a-zA-Z0-9\s'!?]", " ", text)

def remove_stopwords_safe(text):
    return " ".join(w for w in text.split() if w not in safe_stop_words)

def lemmatize_text(text):
    return " ".join(lemmatizer.lemmatize(w) for w in text.split())

def basic_token_cleanup(text):
    return re.sub(r"\s+", " ", text).strip()

# ── TextCleaner transformer ──────────────────────────────────────────────────
class TextCleaner(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return [self.clean_data(text) for text in X]

    @staticmethod
    def clean_data(text):
        text = str(text)
        text = fix_unicode_text(text)           # 0. fix encoding
        text = convert_emojis(text)             # 1. emojis → words
        text = html_pattern.sub(" ", text)      # 2. strip HTML
        text = url_pattern.sub(" <url> ", text) # 3. replace URLs
        text = mention_pattern.sub(" <name> ", text)        # 4. replace @mentions
        text = hashtag_pattern.sub(r" \1 ", text)           # 5. strip # keep word
        text = text.lower()                                  # 6. lowercase
        text = expand_contractions(text)                     # 7. contractions
        text = normalize_slang_text(text)                    # 8. slang
        text = re.sub(r"\d+", " <number> ", text)           # 9. numbers
        text = reduce_elongated_words(text)                  # 10. sooo → soo
        text = reduce_repeated_punctuation(text)             # 11. !!! → !
        text = remove_special_characters(text)               # 12. strip symbols
        text = remove_stopwords_safe(text)                   # 13. stopwords
        text = lemmatize_text(text)                          # 14. lemmatize
        text = basic_token_cleanup(text)                     # 15. whitespace
        return text

# ── Data loading & preparation ───────────────────────────────────────────────
df = pd.read_csv("data.csv")
print(df['status'].value_counts())



# =====================================================
# SOLUTION B : Fusion des classes similaires + TOUTES les données
# 7 classes → 4 classes regroupées
#
# Regroupement :
#   "Depression" + "Suicidal"          → "Depression_Suicidal"
#   "Anxiety"    + "Stress"            → "Anxiety_Stress"
#   "Bipolar"    + "Personality disorder" → "Bipolar_Personality"
#   "Normal"                           → "Normal"  (inchangé)
#
# Justification : ces classes partagent des mots très similaires.
# Les fusionner réduit la confusion inter-classes → accuracy attendue 0.85+
# =====================================================

CLASS_MAPPING = {
    "Depression"           : "Depression_Suicidal",
    "Suicidal"             : "Depression_Suicidal",
    "Anxiety"              : "Anxiety_Stress",
    "Stress"               : "Anxiety_Stress",
    "Bipolar"              : "Bipolar_Personality",
    "Personality disorder" : "Bipolar_Personality",
    "Normal"               : "Normal"
}

df["status"] = df["status"].map(CLASS_MAPPING)

print("Dataset chargé et classes fusionnées avec succès.")
print("Dimensions :", df.shape)
print("\nNouvelle distribution des classes :")
display(df["status"].value_counts().to_frame("nombre_exemples"))

label_encoder = LabelEncoder()
df["status"] = label_encoder.fit_transform(df["status"])

df_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df["status"]
)

# ── Pipeline ─────────────────────────────────────────────────────────────────
pipeline = Pipeline([
    ("cleaner", TextCleaner()),
    ("tfidf", TfidfVectorizer(
        token_pattern=r"\b[a-zA-Z']{2,}\b",
        sublinear_tf=True
    )),
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

# ── Train & save ─────────────────────────────────────────────────────────────
pipeline.fit(df_train["statement"], df_train["status"])
joblib.dump(pipeline, "pipeline_finale.joblib")
print("Pipeline saved to pipeline_finale.joblib")

status
Normal                  16351
Depression              15404
Suicidal                10653
Anxiety                  3888
Bipolar                  2877
Stress                   2669
Personality disorder     1201
Name: count, dtype: int64
Dataset chargé et classes fusionnées avec succès.
Dimensions : (53043, 3)

Nouvelle distribution des classes :


,nombre_exemples
status,
Depression_Suicidal,26057
Normal,16351
Anxiety_Stress,6557
Bipolar_Personality,4078


Pipeline saved to pipeline_finale.joblib


In [8]:
joblib.dump(label_encoder, "label_encoder_finale.joblib")  

['label_encoder_finale.joblib']

In [9]:
print("Classes:", label_encoder.classes_)

Classes: ['Anxiety_Stress' 'Bipolar_Personality' 'Depression_Suicidal' 'Normal']
